In [1]:
import os, glob
import sounddevice as sd
from scipy.io.wavfile import write
from src.constants import Constants as C
from src.levenshtein import damerau_levenshtein_weighted
from src.parsers import wav_to_logmel
from src.evaluator import evaluate_word
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from src.constants import Constants as C
from pathlib import Path

from src.parsers import PhonemeWindowDataset
from src.NeuralModel import CRNN
from src.trainers import train_model, evaluate_tm, load_checkpoint
from src.evaluator import evaluate_audio
from src.wordmaker import PHONEME_TO_LETTERS, levenshtein_distance, phonemes_to_text, parse_words, WLIST1000, proba_predict
 


CHECKPOINT_PATH = "../trained_models/BetterDataSoft.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN()
meta = load_checkpoint(CHECKPOINT_PATH, model, device=device)
model.eval()

print("checkpoint meta keys:", list(meta.keys()))

checkpoint meta keys: []


In [5]:
duration = 2  # seconds
frequency = C.SAMPLE_RATE  # sample rate
recording = sd.rec(int(duration * frequency), samplerate=frequency, channels=1)
print("recording started")
sd.wait()
write("../recordings/output.wav", frequency, recording) 
print("recording ended")

result = evaluate_audio(
    "../recordings/output.wav",
    model=model,
    device=device,
    show_per_window=False,
    top_k=3,
)
w = proba_predict(result, p=0.67, longer_reg = True, aeo_reg=True, )
wtext = phonemes_to_text(w, after_silence=False)
output = WLIST1000[0]
mindist = 1000
for wr in WLIST1000:
    #print(f"Testing word: {wr}")
    dist = damerau_levenshtein_weighted(wr, w)
    #print(dist)
    if dist < mindist:
        mindist = dist
        output = wr
print("_______________________________________________________________")
print(f"Prediction without correction: {wtext}")
print(f"Best match:  ---- {output} ----- with distance {mindist}")

recording started
recording ended

file: ../recordings/output.wav
duration: 2.01s, 97 windows

predicted phoneme sequence (count, avg prob):
  sil(55,0.93) p(3,0.52) sil(1,0.25) j(2,0.26) n(1,0.51) i(5,0.64) e(4,0.68) a(1,0.39) e(1,0.50) z(9,0.70) r(1,0.24) e(2,0.44) a(1,0.47) e(5,0.47) k(3,0.72) sil(2,0.53) k(1,0.57)

average confidence on predicted class: 0.774
_______________________________________________________________
Prediction without correction: pniezek
Best match:  ---- piesek ----- with distance 1.9000000000000001


In [21]:
import tkinter as tk
from tkinter import ttk
import threading


class PhonemeRecognizerApp:
    def __init__(self, root, model, device, word_list, 
                 duration=2.0, sample_rate=16000,
                 evaluate_audio_fn=None, proba_predict_fn=None,
                 phonemes_to_text_fn=None, distance_fn=None):
        self.root = root
        self.model = model
        self.device = device
        self.word_list = word_list
        self.duration = duration
        self.sample_rate = sample_rate
        
        self.evaluate_audio = evaluate_audio_fn
        self.proba_predict = proba_predict_fn
        self.phonemes_to_text = phonemes_to_text_fn
        self.distance_fn = distance_fn
        
        self.recording_path = '../recordings/output.wav'
        os.makedirs(os.path.dirname(self.recording_path), exist_ok=True)
        
        self._build_ui()
    
    def _build_ui(self):
        self.root.title('Phoneme Recognizer')
        self.root.geometry('1000x850')
        self.root.tk.call('tk', 'scaling', 2.0)
    
    
        self.root.configure(bg='#1e1e1e')
        
        # ── Główny kontener ──
        main_frame = tk.Frame(self.root, bg='#1e1e1e', padx=60, pady=40)
        main_frame.pack(fill='both', expand=True)
        
        # ── Tytuł ──
        title = tk.Label(
            main_frame, text='🎙️  Phoneme Recognizer',
            bg='#1e1e1e', fg='#ffffff',
            font=('Arial', 36, 'bold')
        )
        title.pack(pady=(0, 10))
        
        subtitle = tk.Label(
            main_frame, text='Naciśnij SPACJA albo kliknij przycisk',
            bg='#1e1e1e', fg='#888888',
            font=('Arial', 14, 'italic')
        )
        subtitle.pack(pady=(0, 30))
        
        # ── Status ──
        self.status_label = tk.Label(
            main_frame, text='● Gotowy',
            bg='#1e1e1e', fg='#66cc66',
            font=('Arial', 22, 'bold')
        )
        self.status_label.pack(pady=(0, 25))
        
        # ── Wielki przycisk nagrywania ──
        self.record_button = tk.Button(
            main_frame, text='🎤  NAGRAJ',
            command=self._start_recording,
            bg='#cc4444', fg='white',
            font=('Arial', 64, 'bold'),
            width=40, height=10,
            relief='raised', bd=5,
            activebackground='#aa3333',
            cursor='hand2',
        )
        self.record_button.pack(pady=15)
        
        # ── Progress bar ──
        # Custom style większego paska
        style = ttk.Style()
        style.theme_use('default')
        style.configure(
            'Big.Horizontal.TProgressbar',
            troughcolor='#3b3b3b',
            background='#cc4444',
            thickness=30,
        )
        self.progress = ttk.Progressbar(
            main_frame, orient='horizontal',
            length=700, mode='determinate',
            style='Big.Horizontal.TProgressbar',
        )
        self.progress.pack(pady=20)
        
        # ── Sekcja wyników ──
        result_frame = tk.Frame(main_frame, bg='#1e1e1e')
        result_frame.pack(fill='x', pady=25)
        
        # Fonemy
        tk.Label(
            result_frame, text='Fonemy:',
            bg='#1e1e1e', fg='#888888',
            font=('Arial', 16, 'bold')
        ).pack(anchor='w', pady=(0, 5))
        
        self.phonemes_label = tk.Label(
            result_frame, text='—',
            bg='#2d2d2d', fg='#ffffff',
            font=('Courier', 20, 'bold'),
            padx=20, pady=15, anchor='w',
            relief='flat', bd=0,
        )
        self.phonemes_label.pack(fill='x', pady=(0, 15))
        
        # Transkrypcja
        tk.Label(
            result_frame, text='Transkrypcja:',
            bg='#1e1e1e', fg='#888888',
            font=('Arial', 16, 'bold')
        ).pack(anchor='w', pady=(0, 5))
        
        self.text_label = tk.Label(
            result_frame, text='—',
            bg='#2d2d2d', fg='#ffffff',
            font=('Arial', 22),
            padx=20, pady=15, anchor='w',
            relief='flat', bd=0,
        )
        self.text_label.pack(fill='x', pady=(0, 15))
        
        # Najlepsze dopasowanie — duże, wyróżnione
        tk.Label(
            result_frame, text='💡  Najlepsze dopasowanie:',
            bg='#1e1e1e', fg='#888888',
            font=('Arial', 16, 'bold')
        ).pack(anchor='w', pady=(0, 5))
        
        self.match_label = tk.Label(
            result_frame, text='—',
            bg='#2d5d2d', fg='#ffffff',
            font=('Arial', 42, 'bold'),
            padx=30, pady=25, anchor='center',
            relief='flat', bd=0,
        )
        self.match_label.pack(fill='x', pady=(0, 5))
        
        self.distance_label = tk.Label(
            result_frame, text='',
            bg='#1e1e1e', fg='#888888',
            font=('Arial', 12, 'italic')
        )
        self.distance_label.pack(anchor='e', pady=(0, 5))
        
        # ── Hotkeys ──
        self.root.bind('<space>', lambda e: self._start_recording())
        self.root.bind('<Return>', lambda e: self._start_recording())
    
    def _start_recording(self):
        # Nie pozwól zacząć nowego nagrywania jeśli już trwa
        if self.record_button['state'] == 'disabled':
            return
        
        self.record_button.configure(state='disabled', bg='#666666', text='⏺  NAGRYWAM...')
        self.status_label.configure(text='🔴 Mów teraz...', fg='#ff6666')
        
        # Reset wyników
        self.phonemes_label.configure(text='—')
        self.text_label.configure(text='—')
        self.match_label.configure(text='—')
        self.distance_label.configure(text='')
        
        self.progress['value'] = 0
        self._animate_progress()
        
        threading.Thread(target=self._record_and_process, daemon=True).start()
    
    def _animate_progress(self):
        update_interval = 30
        steps = int(self.duration * 1000 / update_interval)
        step_value = 100 / steps
        
        def update(step):
            if step <= steps:
                self.progress['value'] = step * step_value
                self.root.after(update_interval, update, step + 1)
        
        update(0)
    
    def _record_and_process(self):
        try:
            recording = sd.rec(
                int(self.duration * self.sample_rate),
                samplerate=self.sample_rate,
                channels=1,
            )
            sd.wait()
            write(self.recording_path, self.sample_rate, recording)
            
            self.root.after(0, self._update_status, '⚙️ Przetwarzanie...', '#ffaa66')
            
            result = self.evaluate_audio(
                self.recording_path,
                model=self.model,
                device=self.device,
                show_per_window=False,
                top_k=3,
            )
            
            phonemes = self.proba_predict(result, p=0.67, longer_reg=True, aeo_reg=True)
            text = self.phonemes_to_text(phonemes, after_silence=False)
            
            output = self.word_list[0]
            min_dist = float('inf')
            for word in self.word_list:
                dist = self.distance_fn(word, phonemes)
                if dist < min_dist:
                    min_dist = dist
                    output = word
            
            self.root.after(0, self._show_results, phonemes, text, output, min_dist)
            
        except Exception as e:
            self.root.after(0, self._show_error, str(e))
    
    def _update_status(self, text, color):
        self.status_label.configure(text=text, fg=color)
    
    def _show_results(self, phonemes, text, match, distance):
        self.phonemes_label.configure(text=' '.join(phonemes))
        self.text_label.configure(text=text)
        self.match_label.configure(text=match.upper())
        self.distance_label.configure(text=f'distance: {distance:.2f}')
        
        self.status_label.configure(text='● Gotowy', fg='#66cc66')
        self.record_button.configure(state='normal', bg='#cc4444', text='🎤  NAGRAJ')
        self.progress['value'] = 100
    
    def _show_error(self, error_msg):
        self.status_label.configure(text=f'❌ Błąd: {error_msg[:60]}', fg='#ff6666')
        self.record_button.configure(state='normal', bg='#cc4444', text='🎤  NAGRAJ')
        self.progress['value'] = 0


# ─── Użycie ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    root = tk.Tk()
    app = PhonemeRecognizerApp(
        root,
        model=model,
        device=device,
        word_list=WLIST1000,
        duration=2.0,
        sample_rate=C.SAMPLE_RATE,
        evaluate_audio_fn=evaluate_audio,
        proba_predict_fn=proba_predict,
        phonemes_to_text_fn=phonemes_to_text,
        distance_fn=damerau_levenshtein_weighted,
    )
    root.mainloop()
  
    # ... reszta bez zmian ...


file: ../recordings/output.wav
duration: 2.01s, 97 windows

predicted phoneme sequence (count, avg prob):
  sil(15,0.90) p(1,0.40) v(2,0.34) u(2,0.64) v(2,0.32) n(3,0.60) m(3,0.72) a(2,0.64) o(3,0.46) p(2,0.39) t(3,0.54) i2(1,0.43) u(2,0.41) i2(4,0.76) k(1,0.37) p(2,0.45) k(3,0.38) r(1,0.39) o(5,0.67) a(3,0.42) e(4,0.47) a(1,0.32) sil(32,0.92)

average confidence on predicted class: 0.711

file: ../recordings/output.wav
duration: 2.01s, 97 windows

predicted phoneme sequence (count, avg prob):
  sil(20,0.94) i(7,0.71) n(2,0.55) s(3,0.36) f(3,0.54) o(2,0.61) r(3,0.66) n(1,0.33) m(3,0.77) a(2,0.49) d(1,0.30) t(4,0.59) i2(4,0.64) e(1,0.45) i2(1,0.42) k(1,0.49) g(1,0.48) k(3,0.76) a(2,0.35) o(1,0.43) a(5,0.78) m(1,0.30) sil(26,0.92)

average confidence on predicted class: 0.753

file: ../recordings/output.wav
duration: 2.01s, 97 windows

predicted phoneme sequence (count, avg prob):
  sil(8,0.83) m(9,0.71) a(4,0.59) t(4,0.52) e(6,0.62) m(2,0.87) a(3,0.58) t(7,0.47) i2(6,0.66) k(2,0.37) t(